# Movie Review Sentiment Analysis — Integrated four-member experiments
Run this notebook top to bottom in Google Colab with `IMDB Dataset.csv`. It integrates the exact model experiment cells from the four standalone notebooks; the individual files do not need to be executed first.

Use scikit-learn **1.6.1**, matching the existing `requirements.txt`. If needed, install it with `%pip install scikit-learn==1.6.1 wordcloud` in a separate setup cell and restart the runtime before running.

**Preserved:** original loading, EDA, cleaning, word clouds, outer 80/20 split, TF-IDF settings, comparison/evaluation cells, sample-review predictions and three-file frontend export. **Expanded:** the shared training cell is replaced by four independent tuning/evaluation sections plus results assembly. Validation uses an additional training-only split and separate vectorizer.

The final download cell writes `tfidf_vectorizer.joblib`, `all_models.joblib`, and `accuracies.joblib` with the same model-name keys expected by `app.py`. Replace all three files together after a successful complete run. No frontend edits are needed. Scores may change after tuning; update report numbers using the newly computed results.

Outputs are intentionally cleared. Run the full dataset to obtain your results; no full-dataset tuned scores are claimed in this file.

## 1. Import libraries

In [ ]:
import pandas as pd, numpy as np, re, joblib, sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
sns.set_style("whitegrid")

## 2. Load the dataset


In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("IMDB Dataset.csv")
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Basic information

In [ ]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate reviews:", df.duplicated().sum())
df.info()

In [ ]:
# Remove duplicate reviews
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

### 3.2 Class balance
Checking if positive and negative reviews are equal in number.

In [ ]:
print(df["sentiment"].value_counts())
sns.countplot(x="sentiment", data=df, palette="Set2")
plt.title("Positive vs Negative reviews")
plt.show()

### 3.3 Review length
Do positive and negative reviews differ in length?

In [ ]:
df["word_count"] = df["review"].apply(lambda x: len(x.split()))
print(df.groupby("sentiment")["word_count"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="word_count", hue="sentiment", bins=50, kde=True)
plt.xlim(0, 1000)
plt.title("Distribution of review length (words)")
plt.show()

## 4. Text preprocessing
Remove HTML tags (like `<br />`) and convert to lowercase. Stopwords are removed later by TF-IDF.

In [ ]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    return text.lower()

df["clean_review"] = df["review"].apply(clean_text)
df[["review", "clean_review"]].head(3)

### 4.1 Word clouds
Most frequent words in positive and negative reviews.

In [ ]:
from wordcloud import WordCloud, STOPWORDS
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i, label in enumerate(["positive", "negative"]):
    text = " ".join(df[df["sentiment"] == label]["clean_review"])
    wc = WordCloud(width=600, height=400, background_color="white",
                   stopwords=STOPWORDS.union({"br", "movie", "film"})).generate(text)
    ax[i].imshow(wc)
    ax[i].set_title(label.capitalize() + " reviews")
    ax[i].axis("off")
plt.show()

## 5. Feature extraction using TF-IDF
TF-IDF converts text into numbers. We split the data 80% train / 20% test.

In [ ]:
y = (df["sentiment"] == "positive").astype(int)   # positive = 1, negative = 0
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], y, test_size=0.2, random_state=42, stratify=y)

vec = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
Xtr = vec.fit_transform(X_train)
Xte = vec.transform(X_test)
print("Train shape:", Xtr.shape, " Test shape:", Xte.shape)

## Validation setup for the four individual experiments
The original 80/20 split and final TF-IDF matrices above remain unchanged. For tuning only, split the 80% training reviews again into 75% subtraining and 25% validation (60/20/20 overall). A separate vectorizer learns only from subtraining text. Select by validation accuracy, then refit the selected configuration on the original full 80% training matrix. Never choose settings using test results. The final exported models all use the original `vec`.

In [ ]:
from sklearn.base import clone
from time import perf_counter
from IPython.display import display

X_subtrain, X_val, y_subtrain, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)
tuning_vec = clone(vec)
Xsub = tuning_vec.fit_transform(X_subtrain)
Xval = tuning_vec.transform(X_val)
models, preds, accs = {}, {}, {}
results, timing_results, experiment_tables = [], [], {}
selected_settings = {}
print('Subtraining:', Xsub.shape, 'Validation:', Xval.shape)
print('Final training:', Xtr.shape, 'Untouched test:', Xte.shape)

## Model 1: Logistic Regression
**Individual contribution:** baseline, three-setting validation experiment, training/validation graph, selected-model test evaluation, confusion matrix, and incorrect-review examples.

Weighted text features; C controls inverse regularisation strength.

The original setting is included among the candidates. Improvements are not guaranteed; interpret the measured results. All displayed results are generated when you run the cells.

In [ ]:
model_name = 'Logistic Regression'
parameter_name = 'C'
candidate_values = [0.1, 1.0, 10.0]
baseline_value = 1.0

def make_candidate(value):
    return LogisticRegression(C=value, max_iter=1000)

trial_rows = []
for value in candidate_values:
    candidate = make_candidate(value)
    started = perf_counter()
    candidate.fit(Xsub, y_subtrain)
    elapsed = perf_counter() - started
    train_pred = candidate.predict(Xsub)
    val_pred = candidate.predict(Xval)
    trial_rows.append({
        parameter_name: value,
        'Training accuracy': accuracy_score(y_subtrain, train_pred),
        'Validation accuracy': accuracy_score(y_val, val_pred),
        'Validation F1': f1_score(y_val, val_pred),
        'Training seconds': elapsed,
        'Original setting': value == baseline_value
    })
trials = pd.DataFrame(trial_rows)
experiment_tables[model_name] = trials.copy()
display(trials.round(4))
# First candidate wins an exact tie; no test scores enter this decision.
best_index = int(np.argmax(trials['Validation accuracy'].to_numpy()))
best_value = candidate_values[best_index]
selected_settings[model_name] = {parameter_name: best_value}
print('Selected using validation accuracy:', selected_settings[model_name])

In [ ]:
positions = np.arange(len(candidate_values))
plt.figure(figsize=(7, 4))
plt.plot(positions, trials['Training accuracy'], marker='o', label='Training')
plt.plot(positions, trials['Validation accuracy'], marker='o', label='Validation')
plt.xticks(positions, [str(v) for v in candidate_values])
plt.xlabel(parameter_name)
plt.ylabel('Accuracy')
plt.title(model_name + ': training vs validation')
plt.legend()
plt.tight_layout()
plt.show()

### Refit the selected configuration and evaluate once on the test set
Refitting uses the original full-training `Xtr` and its original vectorizer. Metrics below describe the selected model on held-out test reviews, not the tuning subset.

In [ ]:
final_model = make_candidate(best_value)
started = perf_counter()
final_model.fit(Xtr, y_train)
fit_seconds = perf_counter() - started
started = perf_counter()
p = final_model.predict(Xte)
predict_seconds = perf_counter() - started
models[model_name] = final_model
preds[model_name] = p
accs[model_name] = round(accuracy_score(y_test, p), 4)
results.append([model_name, accuracy_score(y_test, p),
                precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p)])
timing_results.append({'Model': model_name, 'Training seconds': fit_seconds,
                       'Test prediction seconds': predict_seconds})
display(pd.DataFrame([results[-1]], columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1-score']).round(4))
print(classification_report(y_test, p, target_names=['Negative', 'Positive']))
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, p, labels=[0, 1]), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
plt.title(model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Positional arrays avoid index alignment issues after the split.
error_table = pd.DataFrame({
    'Review': X_test.to_numpy(), 'Actual': y_test.to_numpy(), 'Predicted': p})
errors = error_table[error_table['Actual'] != error_table['Predicted']].copy()
errors['Actual'] = errors['Actual'].map({0: 'Negative', 1: 'Positive'})
errors['Predicted'] = errors['Predicted'].map({0: 'Negative', 1: 'Positive'})
print('Incorrect predictions:', len(errors), 'out of', len(y_test))
with pd.option_context('display.max_colwidth', 350):
    display(errors.head(5))

### Explain your findings
After running, state: (1) which setting won on validation; (2) whether training and validation scores diverged; (3) final test metrics; (4) patterns you actually observe in the incorrect reviews. Do not claim an improvement or a cause of errors without evidence. Do not retune after reading test errors. Timing depends on the runtime hardware.

## Model 2: Naive Bayes
**Individual contribution:** baseline, three-setting validation experiment, training/validation graph, selected-model test evaluation, confusion matrix, and incorrect-review examples.

Probabilistic text classification; alpha controls additive smoothing.

The original setting is included among the candidates. Improvements are not guaranteed; interpret the measured results. All displayed results are generated when you run the cells.

In [ ]:
model_name = 'Naive Bayes'
parameter_name = 'alpha'
candidate_values = [0.1, 0.5, 1.0]
baseline_value = 1.0

def make_candidate(value):
    return MultinomialNB(alpha=value)

trial_rows = []
for value in candidate_values:
    candidate = make_candidate(value)
    started = perf_counter()
    candidate.fit(Xsub, y_subtrain)
    elapsed = perf_counter() - started
    train_pred = candidate.predict(Xsub)
    val_pred = candidate.predict(Xval)
    trial_rows.append({
        parameter_name: value,
        'Training accuracy': accuracy_score(y_subtrain, train_pred),
        'Validation accuracy': accuracy_score(y_val, val_pred),
        'Validation F1': f1_score(y_val, val_pred),
        'Training seconds': elapsed,
        'Original setting': value == baseline_value
    })
trials = pd.DataFrame(trial_rows)
experiment_tables[model_name] = trials.copy()
display(trials.round(4))
# First candidate wins an exact tie; no test scores enter this decision.
best_index = int(np.argmax(trials['Validation accuracy'].to_numpy()))
best_value = candidate_values[best_index]
selected_settings[model_name] = {parameter_name: best_value}
print('Selected using validation accuracy:', selected_settings[model_name])

In [ ]:
positions = np.arange(len(candidate_values))
plt.figure(figsize=(7, 4))
plt.plot(positions, trials['Training accuracy'], marker='o', label='Training')
plt.plot(positions, trials['Validation accuracy'], marker='o', label='Validation')
plt.xticks(positions, [str(v) for v in candidate_values])
plt.xlabel(parameter_name)
plt.ylabel('Accuracy')
plt.title(model_name + ': training vs validation')
plt.legend()
plt.tight_layout()
plt.show()

### Refit the selected configuration and evaluate once on the test set
Refitting uses the original full-training `Xtr` and its original vectorizer. Metrics below describe the selected model on held-out test reviews, not the tuning subset.

In [ ]:
final_model = make_candidate(best_value)
started = perf_counter()
final_model.fit(Xtr, y_train)
fit_seconds = perf_counter() - started
started = perf_counter()
p = final_model.predict(Xte)
predict_seconds = perf_counter() - started
models[model_name] = final_model
preds[model_name] = p
accs[model_name] = round(accuracy_score(y_test, p), 4)
results.append([model_name, accuracy_score(y_test, p),
                precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p)])
timing_results.append({'Model': model_name, 'Training seconds': fit_seconds,
                       'Test prediction seconds': predict_seconds})
display(pd.DataFrame([results[-1]], columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1-score']).round(4))
print(classification_report(y_test, p, target_names=['Negative', 'Positive']))
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, p, labels=[0, 1]), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
plt.title(model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Positional arrays avoid index alignment issues after the split.
error_table = pd.DataFrame({
    'Review': X_test.to_numpy(), 'Actual': y_test.to_numpy(), 'Predicted': p})
errors = error_table[error_table['Actual'] != error_table['Predicted']].copy()
errors['Actual'] = errors['Actual'].map({0: 'Negative', 1: 'Positive'})
errors['Predicted'] = errors['Predicted'].map({0: 'Negative', 1: 'Positive'})
print('Incorrect predictions:', len(errors), 'out of', len(y_test))
with pd.option_context('display.max_colwidth', 350):
    display(errors.head(5))

### Explain your findings
After running, state: (1) which setting won on validation; (2) whether training and validation scores diverged; (3) final test metrics; (4) patterns you actually observe in the incorrect reviews. Do not claim an improvement or a cause of errors without evidence. Do not retune after reading test errors. Timing depends on the runtime hardware.

## Model 3: Linear SVM
**Individual contribution:** baseline, three-setting validation experiment, training/validation graph, selected-model test evaluation, confusion matrix, and incorrect-review examples.

Linear separating boundary; C controls the penalty for margin violations.

The original setting is included among the candidates. Improvements are not guaranteed; interpret the measured results. All displayed results are generated when you run the cells.

In [ ]:
model_name = 'Linear SVM'
parameter_name = 'C'
candidate_values = [0.1, 1.0, 10.0]
baseline_value = 1.0

def make_candidate(value):
    return LinearSVC(C=value, random_state=42)

trial_rows = []
for value in candidate_values:
    candidate = make_candidate(value)
    started = perf_counter()
    candidate.fit(Xsub, y_subtrain)
    elapsed = perf_counter() - started
    train_pred = candidate.predict(Xsub)
    val_pred = candidate.predict(Xval)
    trial_rows.append({
        parameter_name: value,
        'Training accuracy': accuracy_score(y_subtrain, train_pred),
        'Validation accuracy': accuracy_score(y_val, val_pred),
        'Validation F1': f1_score(y_val, val_pred),
        'Training seconds': elapsed,
        'Original setting': value == baseline_value
    })
trials = pd.DataFrame(trial_rows)
experiment_tables[model_name] = trials.copy()
display(trials.round(4))
# First candidate wins an exact tie; no test scores enter this decision.
best_index = int(np.argmax(trials['Validation accuracy'].to_numpy()))
best_value = candidate_values[best_index]
selected_settings[model_name] = {parameter_name: best_value}
print('Selected using validation accuracy:', selected_settings[model_name])

In [ ]:
positions = np.arange(len(candidate_values))
plt.figure(figsize=(7, 4))
plt.plot(positions, trials['Training accuracy'], marker='o', label='Training')
plt.plot(positions, trials['Validation accuracy'], marker='o', label='Validation')
plt.xticks(positions, [str(v) for v in candidate_values])
plt.xlabel(parameter_name)
plt.ylabel('Accuracy')
plt.title(model_name + ': training vs validation')
plt.legend()
plt.tight_layout()
plt.show()

### Refit the selected configuration and evaluate once on the test set
Refitting uses the original full-training `Xtr` and its original vectorizer. Metrics below describe the selected model on held-out test reviews, not the tuning subset.

In [ ]:
final_model = make_candidate(best_value)
started = perf_counter()
final_model.fit(Xtr, y_train)
fit_seconds = perf_counter() - started
started = perf_counter()
p = final_model.predict(Xte)
predict_seconds = perf_counter() - started
models[model_name] = final_model
preds[model_name] = p
accs[model_name] = round(accuracy_score(y_test, p), 4)
results.append([model_name, accuracy_score(y_test, p),
                precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p)])
timing_results.append({'Model': model_name, 'Training seconds': fit_seconds,
                       'Test prediction seconds': predict_seconds})
display(pd.DataFrame([results[-1]], columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1-score']).round(4))
print(classification_report(y_test, p, target_names=['Negative', 'Positive']))
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, p, labels=[0, 1]), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
plt.title(model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Positional arrays avoid index alignment issues after the split.
error_table = pd.DataFrame({
    'Review': X_test.to_numpy(), 'Actual': y_test.to_numpy(), 'Predicted': p})
errors = error_table[error_table['Actual'] != error_table['Predicted']].copy()
errors['Actual'] = errors['Actual'].map({0: 'Negative', 1: 'Positive'})
errors['Predicted'] = errors['Predicted'].map({0: 'Negative', 1: 'Positive'})
print('Incorrect predictions:', len(errors), 'out of', len(y_test))
with pd.option_context('display.max_colwidth', 350):
    display(errors.head(5))

### Explain your findings
After running, state: (1) which setting won on validation; (2) whether training and validation scores diverged; (3) final test metrics; (4) patterns you actually observe in the incorrect reviews. Do not claim an improvement or a cause of errors without evidence. Do not retune after reading test errors. Timing depends on the runtime hardware.

## Model 4: Decision Tree
**Individual contribution:** baseline, three-setting validation experiment, training/validation graph, selected-model test evaluation, confusion matrix, and incorrect-review examples.

Feature-threshold decisions; maximum depth limits tree complexity.

The original setting is included among the candidates. Improvements are not guaranteed; interpret the measured results. All displayed results are generated when you run the cells.

In [ ]:
model_name = 'Decision Tree'
parameter_name = 'max_depth'
candidate_values = [10, 25, 50]
baseline_value = 50

def make_candidate(value):
    return DecisionTreeClassifier(max_depth=value, random_state=42)

trial_rows = []
for value in candidate_values:
    candidate = make_candidate(value)
    started = perf_counter()
    candidate.fit(Xsub, y_subtrain)
    elapsed = perf_counter() - started
    train_pred = candidate.predict(Xsub)
    val_pred = candidate.predict(Xval)
    trial_rows.append({
        parameter_name: value,
        'Training accuracy': accuracy_score(y_subtrain, train_pred),
        'Validation accuracy': accuracy_score(y_val, val_pred),
        'Validation F1': f1_score(y_val, val_pred),
        'Training seconds': elapsed,
        'Original setting': value == baseline_value
    })
trials = pd.DataFrame(trial_rows)
experiment_tables[model_name] = trials.copy()
display(trials.round(4))
# First candidate wins an exact tie; no test scores enter this decision.
best_index = int(np.argmax(trials['Validation accuracy'].to_numpy()))
best_value = candidate_values[best_index]
selected_settings[model_name] = {parameter_name: best_value}
print('Selected using validation accuracy:', selected_settings[model_name])

In [ ]:
positions = np.arange(len(candidate_values))
plt.figure(figsize=(7, 4))
plt.plot(positions, trials['Training accuracy'], marker='o', label='Training')
plt.plot(positions, trials['Validation accuracy'], marker='o', label='Validation')
plt.xticks(positions, [str(v) for v in candidate_values])
plt.xlabel(parameter_name)
plt.ylabel('Accuracy')
plt.title(model_name + ': training vs validation')
plt.legend()
plt.tight_layout()
plt.show()

### Refit the selected configuration and evaluate once on the test set
Refitting uses the original full-training `Xtr` and its original vectorizer. Metrics below describe the selected model on held-out test reviews, not the tuning subset.

In [ ]:
final_model = make_candidate(best_value)
started = perf_counter()
final_model.fit(Xtr, y_train)
fit_seconds = perf_counter() - started
started = perf_counter()
p = final_model.predict(Xte)
predict_seconds = perf_counter() - started
models[model_name] = final_model
preds[model_name] = p
accs[model_name] = round(accuracy_score(y_test, p), 4)
results.append([model_name, accuracy_score(y_test, p),
                precision_score(y_test, p), recall_score(y_test, p), f1_score(y_test, p)])
timing_results.append({'Model': model_name, 'Training seconds': fit_seconds,
                       'Test prediction seconds': predict_seconds})
display(pd.DataFrame([results[-1]], columns=[
    'Model', 'Accuracy', 'Precision', 'Recall', 'F1-score']).round(4))
print(classification_report(y_test, p, target_names=['Negative', 'Positive']))
plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_test, p, labels=[0, 1]), annot=True, fmt='d',
            cmap='Blues', xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
plt.title(model_name)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Positional arrays avoid index alignment issues after the split.
error_table = pd.DataFrame({
    'Review': X_test.to_numpy(), 'Actual': y_test.to_numpy(), 'Predicted': p})
errors = error_table[error_table['Actual'] != error_table['Predicted']].copy()
errors['Actual'] = errors['Actual'].map({0: 'Negative', 1: 'Positive'})
errors['Predicted'] = errors['Predicted'].map({0: 'Negative', 1: 'Positive'})
print('Incorrect predictions:', len(errors), 'out of', len(y_test))
with pd.option_context('display.max_colwidth', 350):
    display(errors.head(5))

### Explain your findings
After running, state: (1) which setting won on validation; (2) whether training and validation scores diverged; (3) final test metrics; (4) patterns you actually observe in the incorrect reviews. Do not claim an improvement or a cause of errors without evidence. Do not retune after reading test errors. Timing depends on the runtime hardware.

## Assemble the four models for the original comparison and frontend

In [ ]:
expected_names = ['Logistic Regression', 'Naive Bayes', 'Linear SVM', 'Decision Tree']
assert list(models) == expected_names
assert set(preds) == set(accs) == set(models)
assert all(m.n_features_in_ == Xtr.shape[1] for m in models.values())
results_df = pd.DataFrame(results, columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1-score'])
results_df = results_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
display(results_df.round(4))
display(pd.DataFrame(timing_results).round(4))
print('Selected settings:', selected_settings)

### 6.1 Accuracy comparison

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(x="Accuracy", y="Model", data=results_df, palette="viridis")
plt.xlim(0.7, 1)
for i, v in enumerate(results_df["Accuracy"]):
    plt.text(v + 0.002, i, f"{v*100:.2f}%", va="center")
plt.title("Model accuracy comparison")
plt.show()

## 7. Evaluation

### 7.1 Confusion matrices
Shows how many reviews each model classified correctly and wrongly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

### 7.2 Classification report of the best model

In [ ]:
best = results_df.loc[0, "Model"]
print("Best model:", best)
print(classification_report(y_test, preds[best], target_names=["Negative", "Positive"]))

### 7.3 Most important words (Logistic Regression)
Words the model uses most to decide positive or negative.

In [ ]:
lr = models["Logistic Regression"]
words = np.array(vec.get_feature_names_out())
order = np.argsort(lr.coef_[0])
top_neg, top_pos = words[order[:15]], words[order[-15:]]
print("Top NEGATIVE words:", list(top_neg))
print("Top POSITIVE words:", list(top_pos))

## 8. Test with your own reviews

In [ ]:
def predict_review(text, model_name="Logistic Regression"):
    X = vec.transform([clean_text(text)])
    return "Positive" if models[model_name].predict(X)[0] == 1 else "Negative"

tests = ["This movie was absolutely fantastic, great acting!",
         "Terrible film. Boring and a complete waste of time."]
for t in tests:
    print(t, "->", {n: predict_review(t, n) for n in models})

## 9. Save the models for the Streamlit app
This downloads 3 files. Put them in the same folder as `app.py`.

In [ ]:
joblib.dump(vec, "tfidf_vectorizer.joblib")
joblib.dump(models, "all_models.joblib")
joblib.dump(accs, "accuracies.joblib")
print("scikit-learn version:", sklearn.__version__)

files.download("tfidf_vectorizer.joblib")
files.download("all_models.joblib")
files.download("accuracies.joblib")

## Conclusion
Use the results from this run to identify the best test accuracy, compare error types and discuss the validation experiments. These four models are alternatives, not an ensemble. A single validation split and mostly fixed preprocessing limit the conclusions. Each member owns one model experiment; shared preprocessing and frontend integration connect the contributions.